# Evaluacion independiente sobre el holdout final

Este cuaderno evalua los modelos entrenados con 100.000 y 500.000 estados sobre
las ultimas 52.563 filas de `strategic.db`, que no fueron utilizadas para entrenar
ninguno de los dos checkpoints. La separacion es determinista y se realiza por
posicion en la tabla (`OFFSET = 500000`), porque el dataset no proporciona un
identificador de partida que permita agrupar los estados por partida.

Se conserva exactamente el `MahjongTokenizer`, el `Tensorizer`, el `MahjongCollator`,
la arquitectura y la taxonomia estrategica del proyecto. `torch.inference_mode()`
solo evita almacenar gradientes durante la evaluacion y no modifica las predicciones.

In [ ]:
from pathlib import Path
import gzip
import json
import sqlite3
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis.error_analysis import categorize_mistake
from src.analysis.ukeire import evaluate_discards_ukeire
from src.data.collator import MahjongCollator
from src.data.tokenizer import MahjongTokenizer
from src.features.tensorization import Tensorizer
from src.models.transformer_model import MahjongTransformer
from src.utils.tile_decoder import tile136_to_string

## Configuracion reproducible

Los paths de salida son nuevos para no sobrescribir los analisis anteriores.

In [ ]:
DB_PATH = PROJECT_ROOT / "data/discard/strategic.db"
HOLDOUT_OFFSET = 500_000
BATCH_SIZE = 64

CHECKPOINTS = {
    "100k": PROJECT_ROOT / "checkpoints/strategic_100k/strategic_baseline.pt",
    "500k": PROJECT_ROOT / "checkpoints/strategic_500k/epoch_15.pt",
}

TABLES_DIR = PROJECT_ROOT / "reports/holdout"
FIGURES_DIR = PROJECT_ROOT / "reports/figures/holdout"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

In [ ]:
with sqlite3.connect(DB_PATH) as connection:
    total_states, min_id, max_id = connection.execute(
        "SELECT COUNT(*), MIN(Id), MAX(Id) FROM Discard"
    ).fetchone()

holdout_size = total_states - HOLDOUT_OFFSET
assert total_states == 552_563, f"Se esperaban 552563 filas, hay {total_states}"
assert (min_id, max_id) == (1, total_states), "Los Id no forman una secuencia contigua"
assert holdout_size == 52_563

split_df = pd.DataFrame([
    {"partition": "training_prefix", "start_offset": 0,
     "end_offset_inclusive": HOLDOUT_OFFSET - 1, "states": HOLDOUT_OFFSET},
    {"partition": "independent_holdout", "start_offset": HOLDOUT_OFFSET,
     "end_offset_inclusive": total_states - 1, "states": holdout_size},
])
split_df.to_csv(TABLES_DIR / "holdout_split.csv", index=False)
split_df

## Dataset

Las filas del holdout se leen una vez y en orden de `Id`. La construccion del estado
canonico reproduce literalmente la utilizada por `MahjongDataset`; no se anaden ni
eliminan campos de la representacion que reciben los checkpoints.

In [ ]:
def canonicalize(decoded):
    return {
        "round_wind": decoded["round_wind"],
        "num_honba": decoded["num_honba"],
        "num_riichi": decoded["num_riichi"],
        "player_wind": decoded["player_wind"],
        "position": decoded["position"],
        "remain_tiles": decoded["remain_tiles"],
        "dora_indicators": decoded["dora_indicators"],
        "hand_tiles": decoded["hand_tiles"],
        "players": {str(i): decoded[str(i)] for i in range(4)},
        "valid_actions": decoded["valid_actions"],
        "action_idx": decoded["action_idx"],
    }


class IndependentHoldoutDataset(Dataset):
    def __init__(self, db_path, offset):
        with sqlite3.connect(db_path) as connection:
            self.rows = connection.execute(
                "SELECT Id, Data FROM Discard ORDER BY Id LIMIT -1 OFFSET ?", (offset,)
            ).fetchall()
        self.tokenizer = MahjongTokenizer()
        self.tensorizer = Tensorizer()

    def __len__(self):
        return len(self.rows)

    def state_at(self, index):
        row_id, blob = self.rows[index]
        decoded = json.loads(gzip.decompress(blob))
        return row_id, canonicalize(decoded)

    def __getitem__(self, index):
        row_id, state = self.state_at(index)
        tensors = self.tensorizer.tensorize(self.tokenizer.tokenize(state))
        tensors["target"] = state["action_idx"]
        return row_id, tensors


collator = MahjongCollator()


def collate_holdout(samples):
    row_ids, tensor_samples = zip(*samples)
    return torch.tensor(row_ids), collator.collate(list(tensor_samples))


holdout_dataset = IndependentHoldoutDataset(DB_PATH, HOLDOUT_OFFSET)
holdout_loader = DataLoader(
    holdout_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_holdout,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print(f"Estados independientes: {len(holdout_dataset):,}")

## Evaluacion predictiva de ambos checkpoints

In [ ]:
def move_batch(batch, target_device):
    return {
        key: ([item.to(target_device) for item in value]
              if isinstance(value, list) else value.to(target_device))
        for key, value in batch.items()
    }


def evaluate_checkpoint(label, checkpoint_path):
    model = MahjongTransformer().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    rows = []
    loss_sum = 0.0
    top1_correct = top3_correct = top5_correct = 0

    with torch.inference_mode():
        for row_ids, batch in tqdm(holdout_loader, desc=f"Modelo {label}"):
            batch = move_batch(batch, device)
            targets = batch["target"]
            logits_list = model(batch)

            for row_id, logits, target in zip(row_ids.tolist(), logits_list, targets):
                target_index = int(target.item())
                probabilities = torch.softmax(logits, dim=0)
                prediction = int(torch.argmax(logits).item())
                ranking = torch.argsort(logits, descending=True)

                loss_sum += F.cross_entropy(
                    logits.unsqueeze(0), target.unsqueeze(0), reduction="sum"
                ).item()
                top1_correct += prediction == target_index
                top3_correct += target_index in ranking[:min(3, len(ranking))].tolist()
                top5_correct += target_index in ranking[:min(5, len(ranking))].tolist()
                rows.append({
                    "model": label,
                    "row_id": row_id,
                    "database_offset": row_id - 1,
                    "expert_action_idx": target_index,
                    "model_action_idx": prediction,
                    "model_confidence": float(probabilities[prediction].item()),
                    "num_valid_actions": len(logits),
                })

    n = len(rows)
    metrics = {
        "model": label,
        "checkpoint": str(checkpoint_path.relative_to(PROJECT_ROOT)),
        "evaluated_states": n,
        "loss": loss_sum / n,
        "top1_accuracy": top1_correct / n,
        "top3_accuracy": top3_correct / n,
        "top5_accuracy": top5_correct / n,
    }
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return metrics, pd.DataFrame(rows)


predictive_metrics = []
prediction_frames = []
for model_label, checkpoint_path in CHECKPOINTS.items():
    metrics, predictions = evaluate_checkpoint(model_label, checkpoint_path)
    predictive_metrics.append(metrics)
    prediction_frames.append(predictions)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictive_df = pd.DataFrame(predictive_metrics)
predictive_df

## Evaluacion estrategica

Se calculan Shanten y Ukeire una sola vez por estado y se aplican las mismas
categorias de los notebooks 08 y 09 a la prediccion de cada modelo.

In [ ]:
predictions_by_model = {
    label: frame.set_index("row_id")
    for label, frame in predictions_df.groupby("model")
}
strategic_rows = []

for index in tqdm(range(len(holdout_dataset)), desc="Shanten y Ukeire"):
    row_id, state = holdout_dataset.state_at(index)
    expert_action = state["valid_actions"][state["action_idx"]]
    discard_metrics = {
        result["discard"]: result for result in evaluate_discards_ukeire(state)
    }

    for model_label, model_predictions in predictions_by_model.items():
        prediction = model_predictions.loc[row_id]
        model_action = state["valid_actions"][int(prediction["model_action_idx"])]
        # Los notebooks 08 y 09 solo categorizaban comparaciones entre descartes.
        # Conservamos ese mismo criterio y contabilizamos su cobertura en el resumen.
        if expert_action["type"] != 1 or model_action["type"] != 1:
            continue
        expert_tile = expert_action["tiles"][0]
        model_tile = model_action["tiles"][0]
        expert_metrics = discard_metrics[expert_tile]
        model_metrics = discard_metrics[model_tile]
        result = {
            "model": model_label,
            "row_id": row_id,
            "database_offset": row_id - 1,
            "expert_tile": tile136_to_string(expert_tile),
            "model_tile": tile136_to_string(model_tile),
            "same_action": expert_tile == model_tile,
            "expert_shanten": expert_metrics["shanten"],
            "model_shanten": model_metrics["shanten"],
            "expert_ukeire": expert_metrics["ukeire"],
            "model_ukeire": model_metrics["ukeire"],
            "shanten_difference": expert_metrics["shanten"] - model_metrics["shanten"],
            "ukeire_difference": expert_metrics["ukeire"] - model_metrics["ukeire"],
            "model_confidence": prediction["model_confidence"],
        }
        result["category"] = categorize_mistake(result)
        strategic_rows.append(result)

strategic_df = pd.DataFrame(strategic_rows)

positive_categories = {
    "exact_match", "equivalent", "near_equivalent", "better_ukeire", "better_shanten"
}
strategic_summary = (
    strategic_df.assign(strategic_success=lambda frame: frame["category"].isin(positive_categories))
    .groupby("model", as_index=False)
    .agg(
        strategically_evaluated_states=("strategic_success", "size"),
        strategic_accuracy=("strategic_success", "mean"),
        exact_tile_accuracy=("same_action", "mean"),
        mean_confidence=("model_confidence", "mean"),
        mean_ukeire_difference=("ukeire_difference", "mean"),
    )
)
summary_df = predictive_df.merge(strategic_summary, on="model")
summary_df

## Tablas y figuras finales

In [ ]:
category_order = [
    "exact_match", "equivalent", "near_equivalent", "better_shanten",
    "better_ukeire", "shanten_loss", "ukeire_loss", "unknown",
]
category_distribution = (
    strategic_df.groupby(["model", "category"]).size().rename("count").reset_index()
)
category_distribution["proportion"] = category_distribution.groupby("model")["count"].transform(
    lambda values: values / values.sum()
)

predictions_df.to_csv(TABLES_DIR / "holdout_predictions.csv", index=False)
strategic_df.to_csv(TABLES_DIR / "holdout_strategic_results.csv", index=False)
summary_df.to_csv(TABLES_DIR / "holdout_model_summary.csv", index=False)
category_distribution.to_csv(TABLES_DIR / "holdout_category_distribution.csv", index=False)

sns.set_theme(style="whitegrid")

metric_plot = summary_df.melt(
    id_vars="model",
    value_vars=["top1_accuracy", "top3_accuracy", "top5_accuracy", "strategic_accuracy"],
    var_name="metric",
    value_name="value",
)
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=metric_plot, x="metric", y="value", hue="model")
ax.set(xlabel="Metrica", ylabel="Proporcion", ylim=(0, 1),
       title="Rendimiento en el conjunto de evaluacion independiente")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "holdout_model_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=category_distribution,
    x="category", y="proportion", hue="model", order=category_order,
)
ax.set(xlabel="Categoria estrategica", ylabel="Proporcion",
       title="Distribucion estrategica en el conjunto independiente")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "holdout_category_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.histplot(
    data=strategic_df, x="model_confidence", hue="model",
    bins=30, stat="density", common_norm=False, element="step", fill=False,
)
ax.set(xlabel="Confianza de la prediccion", ylabel="Densidad",
       title="Confianza en el conjunto de evaluacion independiente")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "holdout_confidence_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Interpretacion y limitacion

Estos resultados sustituyen las cifras obtenidas al evaluar sobre los prefijos de
entrenamiento. El holdout no fue visto por ninguno de los dos modelos, pero la
independencia solo puede afirmarse a nivel de filas reservadas. El formato publicado
no contiene un identificador de partida; por tanto, no se puede descartar que estados
de una misma partida aparezcan a ambos lados del corte. El dataset procede de logs de
Tenhou transformados a SQLite y se publica con licencia Apache 2.0.

Fuente de la descripcion y ejemplo de uso:
https://www.kaggle.com/code/hphphp123321/how-to-use